<a href="https://colab.research.google.com/github/Rivaldo2002/Machine-Learning-Image-Classification/blob/main/Machine_Learning_Image_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U albumentations
!pip install tensorflow
!pip install visualkeras
!apt-get install -y fonts-dejavu-extra
!pip install pillow

In [ ]:
!apt-get update
!apt-get install -y fonts-liberation
!apt-get install -y fontconfig
!fc-cache -fv

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from google.colab import drive
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import tensorflow as tf
from functools import partial
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adamax, Adadelta, Adagrad, Ftrl
from tensorflow.keras.metrics import Precision, Recall, F1Score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from tensorflow.keras.utils import plot_model
import visualkeras
from PIL import ImageFont
import random
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
drive.mount('/content/drive')

In [ ]:
base_dir = '/content/drive/MyDrive/TugasAkhir/dataset'
split_dir = '/content/drive/MyDrive/TugasAkhir/split_dataset'
train_dir = '/content/drive/MyDrive/TugasAkhir/train'
test_dir = '/content/drive/MyDrive/TugasAkhir/test'

In [ ]:
train_dir = os.path.join(split_dir, 'train')
test_dir = os.path.join(split_dir, 'test')
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

In [ ]:
base_path = Path('/content/drive/MyDrive/TugasAkhir/dataset')
for class_path in base_path.iterdir():
    if class_path.is_dir():
        class_name = class_path.name
    if not os.path.isdir(class_path):
        continue

    images = [os.path.join(class_path, img) for img in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, img))]

    train_images, test_images = train_test_split(images, test_size=0.2, random_state=42)

    train_class_dir = os.path.join(train_dir, class_name)
    test_class_dir = os.path.join(test_dir, class_name)
    os.makedirs(train_class_dir, exist_ok=True)
    os.makedirs(test_class_dir, exist_ok=True)

    for img in train_images:
        shutil.copy(img, train_class_dir)
    for img in test_images:
        shutil.copy(img, test_class_dir)

In [ ]:
batch_size = 32
image_size = [224,224]
ima_size = 224
autotune = tf.data.AUTOTUNE
input_size = (image_size,image_size,3)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='training',
    batch_size=None,
    seed=123,
    image_size = (ima_size,ima_size)
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='validation',
    batch_size=batch_size,
    seed=123,
    image_size = (ima_size,ima_size)
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=123,
    batch_size=1,
    image_size = (ima_size,ima_size)
)

dataset_class = val_dataset.class_names
dataset_class

In [ ]:
def resize_rescale(image, label):
  return tf.image.resize(image, (224,224))/255.0, label

In [ ]:
val_dataset = val_dataset.map(resize_rescale).prefetch(autotune)

test_dataset = test_dataset.map(resize_rescale).prefetch(autotune)

In [ ]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
transforms = A.Compose([
	      A.HorizontalFlip(p=0.5),
	      A.Rotate(limit=15, p=0.5),
	      A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
	      A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
	      A.OneOf([
	        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
	        A.GaussNoise(var_limit=(1.0, 5.0), p=0.2),
	      ]),
	      A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
	    ])

In [ ]:
def aug_ds(image, img_size):
    image = image / 255.0 if image.dtype == tf.float32 else image
    data = {"image": image}
    aug_data = transforms(**data)
    aug_img = aug_data["image"]
    aug_img = tf.cast(aug_img, tf.float32) / 255.0 if aug_img.dtype == np.uint8 else aug_img
    aug_img = tf.clip_by_value(aug_img, 0.0, 1.0)

    return tf.image.resize(aug_img, size=[img_size, img_size])

In [ ]:
def process_data(image, label, img_size):
    aug_img = tf.numpy_function(func=aug_ds, inp=[image, img_size], Tout=tf.float32)
    return aug_img, label

In [ ]:
train_alb = train_dataset.map(partial(process_data, img_size=ima_size),num_parallel_calls=autotune).prefetch(autotune)

In [ ]:
def set_shapes(img, label, img_shape=(ima_size,ima_size,3)):
    img.set_shape(img_shape)
    label.set_shape([])
    return img, label

In [ ]:
train_alb_aug = train_alb.map(set_shapes,num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)

In [ ]:
print(train_alb_aug)
print(val_dataset)
print(test_dataset)

In [ ]:
def view_image(ds):
    image, label = next(iter(ds))
    image = image.numpy()
    label = label.numpy()

    fig = plt.figure(figsize=(22, 22))
    for i in range(20):
        ax = fig.add_subplot(4, 5, i+1, xticks=[], yticks=[])
        ax.imshow(image[i])
        ax.set_title(f"Label: {label[i]}")

In [ ]:
view_image(train_alb_aug)

In [ ]:
view_image(val_dataset)

In [ ]:
def sparse_to_one_hot(labels, num_classes):
    return tf.one_hot(labels, depth=num_classes)

In [ ]:
class Recall(tf.keras.metrics.Recall):
    def __init__(self, name='recall', **kwargs):
        super(Recall, self).__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = sparse_to_one_hot(y_true, tf.shape(y_pred)[-1])
        y_true = tf.reshape(y_true, tf.shape(y_pred))
        return super().update_state(y_true, y_pred, sample_weight)

In [ ]:
class Precision(tf.keras.metrics.Precision):
    def __init__(self, name='precision', **kwargs):
        super(Precision, self).__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = sparse_to_one_hot(y_true, tf.shape(y_pred)[-1])
        y_true = tf.reshape(y_true, tf.shape(y_pred))
        return super().update_state(y_true, y_pred, sample_weight)

In [ ]:
class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.precision = Precision()
        self.recall = Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        precision = self.precision.result()
        recall = self.recall.result()
        return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))

    def reset_states(self):
        self.precision.reset_state()
        self.recall.reset_state()

In [ ]:
input_size = (224,224,3)
font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 12)
loss_alb = "sparse_categorical_crossentropy"

### Muat VGG

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

###Muat Resnet

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)

In [ ]:
ResnetV2.summary()

###Muat Inception

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

Fully Connected Layer


### 1024

###1024 VGG19

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape= input_size,
    name="VGG19",
    pooling="max"
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###1024 Resnet

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)

In [ ]:
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 1024 InceptionV3

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 512

###512 VGG19

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 512 ResnetV2

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)

In [ ]:
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 512 InceptionV3

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 256

### 256 VGG19

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
# Simpan true labels dan predicted labels
true_labels = []
predicted_labels = []
misclassified_images = []  # Untuk menyimpan gambar yang salah klasifikasi
misclassified_true_labels = []  # True label dari gambar salah klasifikasi
misclassified_predicted_labels = []  # Predicted label dari gambar salah klasifikasi

# Loop untuk memproses data test
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

    # Simpan gambar yang salah klasifikasi
    for img, true_label, pred_label in zip(images, labels.numpy(), np.argmax(predictions, axis=1)):
        if true_label != pred_label:
            misclassified_images.append(img.numpy())
            misclassified_true_labels.append(true_label)
            misclassified_predicted_labels.append(pred_label)

# Hitung confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Menampilkan gambar yang salah klasifikasi dengan koordinat (x, y)
def display_all_misclassified_images_with_coordinates(images, true_labels, predicted_labels, class_names, cm, num_images_per_row=5):
    """
    Display all misclassified images with their true label, predicted label,
    and coordinates (x, y) in the confusion matrix.
    """
    num_images = len(images)
    num_rows = (num_images // num_images_per_row) + (1 if num_images % num_images_per_row != 0 else 0)

    plt.figure(figsize=(15, num_rows * 3))
    for i in range(num_images):
        plt.subplot(num_rows, num_images_per_row, i + 1)

        # Pastikan format gambar [H, W, C]
        img = images[i]
        if img.shape[0] == 3:  # Jika channel berada di posisi pertama
            img = img.transpose(1, 2, 0)  # Ubah ke format [H, W, C]

        plt.imshow(img)  # Tampilkan gambar
        true_label = true_labels[i]
        predicted_label = predicted_labels[i]

        # Ambil nilai confusion matrix
        count = cm[true_label, predicted_label]

        # Tambahkan koordinat (x, y) pada title
        plt.title(f"True: {class_names[true_label]} (Row {true_label})\n"
                  f"Pred: {class_names[predicted_label]} (Col {predicted_label})\nCount: {count}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Panggil fungsi untuk menampilkan gambar salah klasifikasi
print(f"Total misclassified images: {len(misclassified_images)}")
if len(misclassified_images) > 0:
    display_all_misclassified_images_with_coordinates(
        misclassified_images, misclassified_true_labels, misclassified_predicted_labels, dataset_class, cm
    )

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 256 ResnetV2

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### 256 InceptionV3

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### activation

Activation sigmoid

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='sigmoid'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='sigmoid'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

VGG Tanh

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='tanh'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='tanh'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

###VGG19 Leaky Relu

In [ ]:
from tensorflow.keras.layers import Dense, LeakyReLU

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
# Simpan true labels dan predicted labels
true_labels = []
predicted_labels = []
misclassified_images = []  # Untuk menyimpan gambar yang salah klasifikasi
misclassified_true_labels = []  # True label dari gambar salah klasifikasi
misclassified_predicted_labels = []  # Predicted label dari gambar salah klasifikasi

# Loop untuk memproses data test
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

    # Simpan gambar yang salah klasifikasi
    for img, true_label, pred_label in zip(images, labels.numpy(), np.argmax(predictions, axis=1)):
        if true_label != pred_label:
            misclassified_images.append(img.numpy())
            misclassified_true_labels.append(true_label)
            misclassified_predicted_labels.append(pred_label)

# Hitung confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Menampilkan gambar yang salah klasifikasi dengan koordinat (x, y)
def display_all_misclassified_images_with_coordinates(images, true_labels, predicted_labels, class_names, cm, num_images_per_row=5):
    """
    Display all misclassified images with their true label, predicted label,
    and coordinates (x, y) in the confusion matrix.
    """
    num_images = len(images)
    num_rows = (num_images // num_images_per_row) + (1 if num_images % num_images_per_row != 0 else 0)

    plt.figure(figsize=(15, num_rows * 3))
    for i in range(num_images):
        plt.subplot(num_rows, num_images_per_row, i + 1)

        # Pastikan format gambar [H, W, C]
        img = images[i]
        if img.shape[0] == 3:  # Jika channel berada di posisi pertama
            img = img.transpose(1, 2, 0)  # Ubah ke format [H, W, C]

        plt.imshow(img)  # Tampilkan gambar
        true_label = true_labels[i]
        predicted_label = predicted_labels[i]

        # Ambil nilai confusion matrix
        count = cm[true_label, predicted_label]

        # Tambahkan koordinat (x, y) pada title
        plt.title(f"True: {class_names[true_label]} (Row {true_label})\n"
                  f"Pred: {class_names[predicted_label]} (Col {predicted_label})\nCount: {count}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Panggil fungsi untuk menampilkan gambar salah klasifikasi
print(f"Total misclassified images: {len(misclassified_images)}")
if len(misclassified_images) > 0:
    display_all_misclassified_images_with_coordinates(
        misclassified_images, misclassified_true_labels, misclassified_predicted_labels, dataset_class, cm
    )

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Hasil Jelek VGG19 Leaky Relu

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Hasil Jelek VGG19 Swish

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###SwishVGG19

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Resnet 512 Leaky Relu

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
# Simpan true labels dan predicted labels
true_labels = []
predicted_labels = []
misclassified_images = []  # Untuk menyimpan gambar yang salah klasifikasi
misclassified_true_labels = []  # True label dari gambar salah klasifikasi
misclassified_predicted_labels = []  # Predicted label dari gambar salah klasifikasi

# Loop untuk memproses data test
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

    # Simpan gambar yang salah klasifikasi
    for img, true_label, pred_label in zip(images, labels.numpy(), np.argmax(predictions, axis=1)):
        if true_label != pred_label:
            misclassified_images.append(img.numpy())
            misclassified_true_labels.append(true_label)
            misclassified_predicted_labels.append(pred_label)

# Hitung confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Menampilkan gambar yang salah klasifikasi dengan koordinat (x, y)
def display_all_misclassified_images_with_coordinates(images, true_labels, predicted_labels, class_names, cm, num_images_per_row=5):
    """
    Display all misclassified images with their true label, predicted label,
    and coordinates (x, y) in the confusion matrix.
    """
    num_images = len(images)
    num_rows = (num_images // num_images_per_row) + (1 if num_images % num_images_per_row != 0 else 0)

    plt.figure(figsize=(15, num_rows * 3))
    for i in range(num_images):
        plt.subplot(num_rows, num_images_per_row, i + 1)

        # Pastikan format gambar [H, W, C]
        img = images[i]
        if img.shape[0] == 3:  # Jika channel berada di posisi pertama
            img = img.transpose(1, 2, 0)  # Ubah ke format [H, W, C]

        plt.imshow(img)  # Tampilkan gambar
        true_label = true_labels[i]
        predicted_label = predicted_labels[i]

        # Ambil nilai confusion matrix
        count = cm[true_label, predicted_label]

        # Tambahkan koordinat (x, y) pada title
        plt.title(f"True: {class_names[true_label]} (Row {true_label})\n"
                  f"Pred: {class_names[predicted_label]} (Col {predicted_label})\nCount: {count}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Panggil fungsi untuk menampilkan gambar salah klasifikasi
print(f"Total misclassified images: {len(misclassified_images)}")
if len(misclassified_images) > 0:
    display_all_misclassified_images_with_coordinates(
        misclassified_images, misclassified_true_labels, misclassified_predicted_labels, dataset_class, cm
    )

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

In [ ]:
model.save('/content/drive/MyDrive/TugasAkhir/Model/Resnet101V2LeakyFinalFinal_model.keras')

### Resnet 512 Swish

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)

In [ ]:
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))


In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Inception 1024 Leaky Relu

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        loss=loss_alb,
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        metrics=['accuracy', Recall(), Precision(), F1Score()],
        run_eagerly=True
    )

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###Inception 1024 Swish

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        loss=loss_alb,
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        metrics=['accuracy', Recall(), Precision(), F1Score()],
        run_eagerly=True
    )

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
# Simpan true labels dan predicted labels
true_labels = []
predicted_labels = []
misclassified_images = []  # Untuk menyimpan gambar yang salah klasifikasi
misclassified_true_labels = []  # True label dari gambar salah klasifikasi
misclassified_predicted_labels = []  # Predicted label dari gambar salah klasifikasi

# Loop untuk memproses data test
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions, axis=1))

    # Simpan gambar yang salah klasifikasi
    for img, true_label, pred_label in zip(images, labels.numpy(), np.argmax(predictions, axis=1)):
        if true_label != pred_label:
            misclassified_images.append(img.numpy())
            misclassified_true_labels.append(true_label)
            misclassified_predicted_labels.append(pred_label)

# Hitung confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# Menampilkan gambar yang salah klasifikasi dengan koordinat (x, y)
def display_all_misclassified_images_with_coordinates(images, true_labels, predicted_labels, class_names, cm, num_images_per_row=5):
    """
    Display all misclassified images with their true label, predicted label,
    and coordinates (x, y) in the confusion matrix.
    """
    num_images = len(images)
    num_rows = (num_images // num_images_per_row) + (1 if num_images % num_images_per_row != 0 else 0)

    plt.figure(figsize=(15, num_rows * 3))
    for i in range(num_images):
        plt.subplot(num_rows, num_images_per_row, i + 1)

        # Pastikan format gambar [H, W, C]
        img = images[i]
        if img.shape[0] == 3:  # Jika channel berada di posisi pertama
            img = img.transpose(1, 2, 0)  # Ubah ke format [H, W, C]

        plt.imshow(img)  # Tampilkan gambar
        true_label = true_labels[i]
        predicted_label = predicted_labels[i]

        # Ambil nilai confusion matrix
        count = cm[true_label, predicted_label]

        # Tambahkan koordinat (x, y) pada title
        plt.title(f"True: {class_names[true_label]} (Row {true_label})\n"
                  f"Pred: {class_names[predicted_label]} (Col {predicted_label})\nCount: {count}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Panggil fungsi untuk menampilkan gambar salah klasifikasi
print(f"Total misclassified images: {len(misclassified_images)}")
if len(misclassified_images) > 0:
    display_all_misclassified_images_with_coordinates(
        misclassified_images, misclassified_true_labels, misclassified_predicted_labels, dataset_class, cm
    )

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

In [ ]:
model.save('/content/drive/MyDrive/TugasAkhir/Model/InceptionV31024SwishFinal_model.keras')

### Optimizer

###SGD VGG19

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###SGD VGG19 0.00001

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###SGD VGG19 learning rate 0.01

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### SGD Resnet

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### SGD Resnet 0.01

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )


In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###SGD InceptionV3

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        loss=loss_alb,
        optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
        metrics=['accuracy', Recall(), Precision(), F1Score()],
        run_eagerly=True
    )

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)


In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)


In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###SGDInceptionV3 0.01

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        loss=loss_alb,
        optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
        metrics=['accuracy', Recall(), Precision(), F1Score()],
        run_eagerly=True
    )

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)


In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)


In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###Adam VGG19 0.01

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)

In [ ]:
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.03),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Adam Resnet 0.01

In [ ]:
ResnetV2 = tf.keras.applications.ResNet101V2(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="ResnetV2",
)
ResnetV2.summary()

In [ ]:
model_deep_learning = ResnetV2

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128),
        tf.keras.layers.LeakyReLU(alpha=0.01),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###Adam Inception 0.01

In [ ]:
InceptionV3 = tf.keras.applications.InceptionV3(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="InceptionV3",
)

In [ ]:
InceptionV3.summary()

In [ ]:
model_deep_learning = InceptionV3

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 200

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False

In [ ]:
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])

    model.compile(
        loss=loss_alb,
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        metrics=['accuracy', Recall(), Precision(), F1Score()],
        run_eagerly=True
    )

    return model

In [ ]:
model = create_model()
model.summary()

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1,
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

### Hasil Jelek VGG19 SGD

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.0001),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###Hasil Jelek VGG19 SGD 0.01

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)

###Hasil Jelek VGG19 adam 0.01

In [ ]:
VGG19 = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    pooling="max",
    input_shape= input_size,
    name="VGG19",
)
VGG19.summary()

In [ ]:
model_deep_learning = VGG19

In [ ]:
print("Number of layers in the pre trained model: ", len(model_deep_learning.layers))

In [ ]:
fine_tune = 13
for layer in model_deep_learning.layers[:fine_tune]:
    layer.trainable = False
for layer in model_deep_learning.layers[fine_tune:]:
    layer.trainable = True

In [ ]:
def create_model():

    model = tf.keras.models.Sequential([
        model_deep_learning,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(512, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(128, activation='swish'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(3, activation='softmax')
    ])


    model.compile(loss=loss_alb,
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                  metrics=['accuracy', Recall(), Precision(), F1Score()],
                  run_eagerly=True)

    return model

In [ ]:
visualkeras.layered_view(model, legend=True, font=font, show_dimension=True, scale_z=0.15)

In [ ]:
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True, show_trainable=True,)

In [ ]:
model = create_model()
model.summary()

In [ ]:
history = model.fit(
    train_alb_aug,
    validation_data=val_dataset,
    epochs=20,
    verbose=1
    )

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

precision = history.history['precision']
val_precision = history.history['val_precision']

recall = history.history['recall']
val_recall = history.history['val_recall']

f1 = history.history['f1_score']
val_f1 = history.history['val_f1_score']

plt.figure(figsize=(14, 10))

plt.subplot(3, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.title('Training and Validation Accuracy')

plt.subplot(3, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([0, max(max(loss), max(val_loss))])
plt.title('Training and Validation Loss')

plt.subplot(3, 2, 3)
plt.plot(precision, label='Training Precision')
plt.plot(val_precision, label='Validation Precision')
plt.legend(loc='lower right')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.title('Training and Validation Precision')

plt.subplot(3, 2, 4)
plt.plot(recall, label='Training Recall')
plt.plot(val_recall, label='Validation Recall')
plt.legend(loc='lower right')
plt.ylabel('Recall')
plt.ylim([0, 1])
plt.title('Training and Validation Recall')

plt.subplot(3, 2, 5)
plt.plot(f1, label='Training F1-Score')
plt.plot(val_f1, label='Validation F1-Score')
plt.legend(loc='lower right')
plt.ylabel('F1-Score')
plt.ylim([0, 1])
plt.title('Training and Validation F1-Score')

plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
model.evaluate(test_dataset)

In [ ]:
true_labels = []
predicted_labels = []
for images, labels in test_dataset:
    true_labels.extend(labels.numpy())
    predictions = model.predict(images, verbose=0)
    predicted_labels.extend(np.argmax(predictions,axis=1))

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_class, yticklabels=dataset_class)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=dataset_class))

In [ ]:
f1 = f1_score(true_labels, predicted_labels, average='weighted')
print("F1-Score (weighted):", f1)
precision = precision_score(true_labels, predicted_labels, average='weighted')
print("Precision (weighted):", precision)
recall = recall_score(true_labels, predicted_labels, average='weighted')
print("Recall (weighted):", recall)